## Biomedical Corpus Generation

In [ ]:
import os
import tqdm
from datasets import load_dataset

# --- CONFIGURATION ---
OUTPUT_FILE = "vital_slm_500m_raw.txt"


LIMIT_PUBMED_ABSTRACTS = 200_000   
LIMIT_FINEWEB_EDU      = 150_000   
LIMIT_MED_WIKIPEDIA    = 100_000   

def clean_text(text):
    if not text: return ""
    return text.strip().replace("\r", " ").replace("\n\n", "\n")

def save_stream_to_file(f, dataset_iter, limit, source_name, text_formatter):
    print(f" -> ⏳ Processing {source_name} (Target: {limit} docs)...")
    count = 0
    pbar = tqdm.tqdm(total=limit, desc=source_name, unit="doc")

    while count < limit:
        try:
            row = next(dataset_iter)
            content = text_formatter(row)
            if content and len(content) > 100: # Increased minimum length to weed out junk
                f.write(content + "\n<|endoftext|>\n")
                count += 1
                pbar.update(1)
        except StopIteration:
            print(f" -> ⚠️ Source {source_name} ran out of data.")
            break
        except Exception:
            continue
    pbar.close()
    print(f" -> ✅ Saved {count} items from {source_name}.\n")

def save_full_dataset(f, dataset, source_name, text_formatter):
    print(f" -> ⏳ Processing {source_name} (Full Dataset)...")
    count = 0
    for row in tqdm.tqdm(dataset, desc=source_name, unit="doc"):
        try:
            content = text_formatter(row)
            if content and len(content) > 50:
                f.write(content + "\n<|endoftext|>\n")
                count += 1
        except:
            continue
    print(f" -> ✅ Saved {count} items from {source_name}.\n")

# --- MAIN EXECUTION ---
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

print(f"🚀 Starting 500M Token Data Generation (Target: ~1.8 GB)...")
with open(OUTPUT_FILE, "a", encoding="utf-8") as f:

    # 1. Medical Textbooks (NEW: The core "brain" of the 50M model)
    print("[1/7] Downloading Medical Textbooks (Full)...")
    try:
    
        ds_books = load_dataset("MedRAG/textbooks", split="train")
        save_full_dataset(f, ds_books, "Medical Textbooks",
                          lambda r: f"Medical Knowledge: {clean_text(r['content'])}")
    except Exception as e: print(f"⚠️ Error with Textbooks: {e}")

    # 2. Medical Wikipedia (NEW: Vast general medical knowledge)
    print("[2/7] Downloading Medical Wikipedia (Limit: 100k)...")
    try:
        ds_medwiki = load_dataset("MedRAG/wikipedia", split="train", streaming=True)
        save_stream_to_file(f, iter(ds_medwiki), LIMIT_MED_WIKIPEDIA, "Medical Wiki",
                            lambda r: f"Topic: {clean_text(r['title'])}\nInformation: {clean_text(r['content'])}")
    except Exception as e: print(f"⚠️ Error with Medical Wiki: {e}")

    # 3. Massive Medical Chat (NEW: 250k Dialogues for conversational ability)
    print("[3/7] Downloading AI Medical Chatbot (Full 250k)...")
    try:
        ds_ai_chat = load_dataset("ruslanmv/ai-medical-chatbot", split="train")
        save_full_dataset(f, ds_ai_chat, "AI Medical Chat",
                          lambda r: f"Patient: {clean_text(r['Patient'])}\nDoctor: {clean_text(r['Doctor'])}")
    except Exception as e: print(f"⚠️ Error with AI Chatbot: {e}")

    # 4. ChatDoctor (Kept from previous)
    print("[4/7] Downloading ChatDoctor (Full 100k)...")
    try:
        ds_chat = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
        save_full_dataset(f, ds_chat, "ChatDoctor",
                          lambda r: f"Patient: {clean_text(r['input'])}\nDoctor: {clean_text(r['output'])}")
    except Exception as e: print(f"⚠️ Error with ChatDoctor: {e}")

    # 5. PubMed Abstracts (Massively Increased to 200k)
    print("[5/7] Downloading PubMed Abstracts (Limit: 200k)...")
    try:
        ds_pubmed = load_dataset("ccdv/pubmed-summarization", split="train", streaming=True)
        save_stream_to_file(f, iter(ds_pubmed), LIMIT_PUBMED_ABSTRACTS, "PubMed",
                            lambda r: f"Research Abstract: {clean_text(r['article'])}")
    except Exception as e: print(f"⚠️ Error with PubMed: {e}")

    # 6. MedQA (Kept from previous, good for strict QA testing)
    print("[6/7] Downloading MedQA (Full)...")
    try:
        ds_medqa = load_dataset("medalpaca/medical_meadow_medqa", split="train")
        save_full_dataset(f, ds_medqa, "MedQA",
                          lambda r: f"Question: {clean_text(r['input'])}\nAnswer: {clean_text(r['output'])}")
    except Exception as e: print(f"⚠️ Error with MedQA: {e}")

    # 7. FineWeb-Edu (Increased to 150k to glue the grammar together)
    print("[7/7] Downloading FineWeb-Edu (Limit: 150k)...")
    try:
        ds_fw = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
        save_stream_to_file(f, iter(ds_fw), LIMIT_FINEWEB_EDU, "FineWeb-Edu",
                            lambda r: clean_text(r['text']))
    except Exception as e: print(f"⚠️ Error with FineWeb: {e}")

# --- VERIFICATION ---
if os.path.exists(OUTPUT_FILE):
    file_size = os.path.getsize(OUTPUT_FILE)
    size_mb = file_size / (1024 * 1024)
    # Using ~3.8 characters per token as a standard English/Medical approximation
    est_tokens = file_size / 3.8 / 1_000_000

    print(f"\n✅ GENERATION COMPLETE!")


## Tokenization (Custom Tokenizer)

In [ ]:
from tokenizers import ByteLevelBPETokenizer
import os

# --- CONFIGURATION ---
DATA_FILE = "vital_slm_500m_raw.txt"
VOCAB_SIZE = 16384
OUTPUT_DIR = "vital_tokenizer"
MAX_LINES_TO_TRAIN = 2_000_000  

print(f"🚀 Training Tokenizer on a sample of {DATA_FILE}...")
print(f"   Target Vocab Size: {VOCAB_SIZE}")

# 1. Create a Memory-Safe Generator

def batch_iterator(batch_size=10000):
    with open(DATA_FILE, "r", encoding="utf-8") as f:
        batch = []
        count = 0
        for line in f:
            if not line.strip(): continue
            batch.append(line)
            count += 1

            # Yield the chunk and clear memory
            if len(batch) == batch_size:
                yield batch
                batch = []

            # Stop reading to prevent RAM overload
            if count >= MAX_LINES_TO_TRAIN:
                print(f"🛑 Reached {MAX_LINES_TO_TRAIN} lines. Stopping reader to save RAM.")
                break

        # Yield any remaining lines
        if batch:
            yield batch

# 2. Initialize Tokenizer
tokenizer = ByteLevelBPETokenizer()

# 3. Train it! (Using the memory-safe iterator)
tokenizer.train_from_iterator(
    batch_iterator(),
    vocab_size=VOCAB_SIZE,
    min_frequency=5,
    special_tokens=[
        "<|endoftext|>",  # 0: End of text / Separator
        "<|padding|>",    # 1: Padding
        "<|unknown|>",    # 2: Unknown words
        "<|mask|>",       # 3: Masking (optional)
    ]
)

# 4. Save it
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

tokenizer.save_model(OUTPUT_DIR)

print(f"✅ Tokenizer saved to folder: '{OUTPUT_DIR}'")
print("   Files generated: vocab.json, merges.txt")


## Binary Serialization

In [ ]:
import os
import numpy as np
import random
import gc
from tokenizers import ByteLevelBPETokenizer
from tqdm import tqdm

# --- CONFIGURATION ---
INPUT_FILE = "vital_slm_500m_raw.txt"
TOKENIZER_DIR = "vital_tokenizer"
TRAIN_FILE = "train.bin"
VAL_FILE = "val.bin"
TRAIN_RATIO = 0.98
BATCH_SIZE = 10000  # Process 10k lines at once (Super fast, very low RAM)

print(f"📂 Loading Tokenizer from '{TOKENIZER_DIR}'...")
tokenizer = ByteLevelBPETokenizer(
    f"{TOKENIZER_DIR}/vocab.json",
    f"{TOKENIZER_DIR}/merges.txt"
)



# High-Speed Batched Processing
print("🚀 Tokenizing in batches of 10,000...")
open(TRAIN_FILE, 'w').close()
open(VAL_FILE, 'w').close()

train_tokens = 0
val_tokens = 0

with open(INPUT_FILE, 'r', encoding='utf-8') as f_in, \
     open(TRAIN_FILE, 'wb') as f_train, \
     open(VAL_FILE, 'wb') as f_val:

    batch = []

    # Progress bar tracks total lines
    with tqdm(total=total_lines, desc="Processing") as pbar:
        for line in f_in:
            if line.strip():
                batch.append(line)

            # When we hit 10,000 lines, process them all at once!
            if len(batch) >= BATCH_SIZE:
                # tokenizer.encode_batch is multi-threaded in Rust!
                encoded_batch = tokenizer.encode_batch(batch)

                for encoded in encoded_batch:
                    ids = np.array(encoded.ids, dtype=np.uint16)
                    if random.random() < TRAIN_RATIO:
                        f_train.write(ids.tobytes())
                        train_tokens += len(ids)
                    else:
                        f_val.write(ids.tobytes())
                        val_tokens += len(ids)

                pbar.update(len(batch))
                batch = [] # Clear the batch from RAM

        # Process any leftover lines at the very end
        if batch:
            encoded_batch = tokenizer.encode_batch(batch)
            for encoded in encoded_batch:
                ids = np.array(encoded.ids, dtype=np.uint16)
                if random.random() < TRAIN_RATIO:
                    f_train.write(ids.tobytes())
                    train_tokens += len(ids)
                else:
                    f_val.write(ids.tobytes())
                    val_tokens += len(ids)
            pbar.update(len(batch))

print(f"\n✅ SUCCESS! Binary datasets ")

In [ ]:
from tokenizers import ByteLevelBPETokenizer

# Define the folder path containing vocab.json and merges.txt
vocab_path = "/kaggle/input/datasets/aman0419/vital-tokenizer-50m/vocab_50m.json"
merges_path = "/kaggle/input/datasets/aman0419/vital-tokenizer-50m/merges_50m.txt"

# Initialize directly
tokenizer = ByteLevelBPETokenizer(
    vocab=vocab_path,
    merges=merges_path
)


print("✅ Tokenizer loaded successfully!")

In [ ]:
# --- ADD THIS TO CELL 1 (SANITY CHECK) ---
import json

def clean_notebook_metadata(filepath):
    with open(filepath, 'r') as f:
        nb = json.load(f)

    # Remove the problematic widgets metadata that causes KeyError: 'state'
    if 'widgets' in nb.get('metadata', {}):
        nb['metadata'].pop('widgets')
        print("✅ Problematic widget metadata removed.")


In [ ]:
import torch
import numpy as np
import os

# --- CONFIGURATION ---
batch_size = 64
block_size = 256
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 1. LOAD DATA (Memmap Mode) ---

train_path = '/kaggle/input/datasets/aman0419/vitallm-data/train_50m.bin'
val_path = '/kaggle/input/datasets/aman0419/vitallm-data/val_50m.bin'

print(f"📂 Loading memmaps from {train_path} and {val_path}...")

# Check if files exist to avoid crashing later
if not os.path.exists(train_path) or not os.path.exists(val_path):
    print("❌ Error: Binary files not found! Did you run the 'prepare_bin_files.py' script?")
else:
    # Load memory-mapped files (Instant access, low RAM usage)
    train_data = np.memmap(train_path, dtype=np.uint16, mode='r')
    val_data = np.memmap(val_path, dtype=np.uint16, mode='r')
    print(f"✅ Loaded! Train tokens: {len(train_data):,}, Val tokens: {len(val_data):,}")

# --- 2. GET_BATCH FUNCTION ---
def get_batch(split):
    # Select the correct dataset
    data = train_data if split == 'train' else val_data

    # Generate random starting indices
    # We subtract block_size to ensure we don't read off the end of the file
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Stack inputs (x) and targets (y)

    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    # Move to GPU (Asynchronous is faster)
    if device == 'cuda':
        # non_blocking=True allows the GPU to compute while data is still moving
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)

    return x, y

## Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm import tqdm
from contextlib import nullcontext
import os

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias=True, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
        self.eps = eps

    def forward(self, x):
        return F.layer_norm(x, x.shape[-1:], self.weight, self.bias, self.eps)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        
        hidden_dim = 4 * config.n_embd

        # w1: Gate Projection
        self.w1 = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        # w2: Value Projection
        self.w2 = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        # c_proj: Output Projection (Down projection)
        self.c_proj = nn.Linear(hidden_dim, config.n_embd, bias=config.bias)

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # SwiGLU Logic: (SiLU(Gate) * Value) -> Projection
        x = F.silu(self.w1(x)) * self.w2(x)

        # 4. Output projection
        return self.dropout(self.c_proj(x))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class SLMConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

class SLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)
        # Apply special scaled init to the residual projections, c_proj
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generate tokens given a conditioning sequence.
        idx: Tensor of shape (B, T)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
config = SLMConfig(
    vocab_size=16384,   
    block_size=256,     
    n_embd=512,        
    n_head=8,          
    n_layer=10,         
    dropout=0.1,
    bias=True           #
)

print(f"🚀 Initializing Model with config: {config}")
model = SLM(config)


In [ ]:

import torch
@torch.no_grad()  
def estimate_loss():  
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with ctx:
                logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## Model Training

In [ ]:
import time
import math
from contextlib import nullcontext

# --- TWO-SESSION CONFIGURATION ---
RESUME_TRAINING = True
CHECKPOINT_PATH = "/kaggle/input/datasets/aman0419/vitallm-50m/vital_lm_50m_checkpoint.pt"
SAVE_PATH = "/kaggle/working/vital_lm_50m_checkpoint.pt"

# --- 1. HYPERPARAMETERS 
max_iters = 31000
warmup_steps = 1000      
learning_rate = 4e-4     
min_lr = 4e-5            
eval_interval = 500      
eval_iters = 200         
weight_decay = 0.1       

# --- 2. SETUP MIXED PRECISION (Crucial for P100 Speed) ---
dtype = 'float16'
device_type = 'cuda' if 'cuda' in device else 'cpu'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

# --- 3. OPTIMIZER (With Weight Decay Fix) ---

param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

optim_groups = [
    {'params': decay_params, 'weight_decay': weight_decay},
    {'params': nodecay_params, 'weight_decay': 0.0}
]

optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-9)

# --- 4. SCHEDULER (Cosine Decay with Warmup) ---

def get_lr(it):
    # 1) Linear warmup
    if it < warmup_steps:
        return learning_rate * (it + 1) / (warmup_steps + 1)
    # 2) If it > max_iters, return min learning rate
    if it > max_iters:
        return min_lr
    # 3) Cosine decay down to min learning rate
    decay_ratio = (it - warmup_steps) / (max_iters - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

# --- 5. INITIALIZATION ---
model.to(device)
start_iter = 0
best_val_loss = float('inf')
train_loss_list = []
val_loss_list = []
lr_list = []
grad_norm_list = []

if RESUME_TRAINING:
    # IMPORTANT: In Session 2, change this path to wherever you uploaded Session 1's checkpoint
    print(f"📂 Loading full checkpoint from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_iter = checkpoint['iter'] + 1
    best_val_loss = checkpoint['best_val_loss']
    train_loss_list = checkpoint['train_loss_list']
    val_loss_list = checkpoint['val_loss_list']
    lr_list = checkpoint['lr_list']
    grad_norm_list = checkpoint['grad_norm_list']
    print(f"✅ Resumed training from step {start_iter}")
else:
    print("🚀 Starting fresh training run (Session 1)...")

# 4. SESSION LIMITS
session_max_iters = 15500 if not RESUME_TRAINING else max_iters

t0 = time.time()

for iter in range(start_iter, session_max_iters):

    # --- A. SET LEARNING RATE ---
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    lr_list.append(lr)

    # --- B. EVALUATION & SAVING ---
    if iter % eval_interval == 0 and iter > 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {lr:.2e}")

        train_loss_list.append(losses['train'].item())
        val_loss_list.append(losses['val'].item())

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            # 💾 SAVE THE FULL STATE, NOT JUST WEIGHTS
            checkpoint = {
                'iter': iter,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'train_loss_list': train_loss_list,
                'val_loss_list': val_loss_list,
                'lr_list': lr_list,
                'grad_norm_list': grad_norm_list
            }
            torch.save(checkpoint, SAVE_PATH)
            print(f"   -> 💾 Saved Checkpoint (Val Loss: {best_val_loss:.4f})")

    # --- C. TRAINING STEP ---
    xb, yb = get_batch('train')

    with ctx:
        logits, loss = model(xb, yb)

    scaler.scale(loss).backward()

    # 💥 CAPTURE GRADIENT NORM (Fixes the undefined variable in Visualization)
    scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    grad_norm_list.append(norm.item())

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    # --- D. LOGGING SPEED ---
    if iter % 100 == 0:
        t1 = time.time()
        dt = t1 - t0
        t0 = t1
        tps = (batch_size * config.block_size * 100) / dt
        print(f"iter {iter} | loss {loss.item():.4f} | time {dt*1000:.2f}ms | speed {tps:.0f} tok/s")


## Training Performance Analytics

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

# --- 1. DATA PREPARATION ---

# Helper to smooth jittery lines
def smooth(scalars, weight=0.85):
    if len(scalars) == 0: return []
    last = scalars[0]
    smoothed = list()
    for point in scalars:
        smoothed_val = last * weight + (1 - weight) * point
        smoothed.append(smoothed_val)
        last = smoothed_val
    return smoothed

# Safe conversion to CPU list
def to_list(data):
    if isinstance(data, list):
        if len(data) == 0: return []
        if isinstance(data[0], torch.Tensor):
            return [x.cpu().detach().item() for x in data]
        return data
    return data

# Load Data
t_loss = to_list(train_loss_list)
v_loss = to_list(val_loss_list)
lrs = to_list(lr_list)
grads = to_list(grad_norm_list)

# Check if grad_norm_list exists (it might not in the optimized loop)
try:
    grads = to_list(grad_norm_list)
except NameError:
    grads = []

# X-Axis alignment
eval_interval = 500  # Adjust this if you changed your config!
eval_steps = np.arange(0, len(t_loss) * eval_interval, eval_interval)
total_steps = np.arange(0, len(lrs))

# --- GRAPH 1: LOSS CURVE (The Standard View) ---
plt.figure(figsize=(10, 6))
plt.plot(eval_steps, t_loss, 'g-', alpha=0.3, label='Raw Train Loss')
plt.plot(eval_steps, smooth(t_loss), 'g-', linewidth=2.5, label='Smoothed Train Loss')
plt.plot(eval_steps, v_loss, 'r--', linewidth=2.5, label='Validation Loss')
plt.title(f' Main Training Curve (Min Val Loss: {min(v_loss):.4f})', fontsize=14)
plt.xlabel('Training Steps')
plt.ylabel('Cross Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# --- GRAPH 2: PERPLEXITY (The "Intelligence" Score) ---
val_ppl = [np.exp(min(l, 10)) for l in v_loss]  # Cap at 10 to prevent explosion
plt.figure(figsize=(10, 6))
plt.plot(eval_steps, val_ppl, 'orange', linewidth=2.5, marker='o', markersize=4)
plt.title(' Perplexity (Fluency Check)', fontsize=14)
plt.xlabel('Training Steps')
plt.ylabel('Perplexity (Lower = Better)')
plt.grid(True, alpha=0.3)
plt.show()

# --- GRAPH 3: GENERALIZATION GAP (New!) ---
gap = np.array(v_loss) - np.array(t_loss)
plt.figure(figsize=(10, 6))
plt.fill_between(eval_steps, gap, 0, where=(gap>0), color='red', alpha=0.3, label='Overfitting Risk')
plt.fill_between(eval_steps, gap, 0, where=(gap<=0), color='blue', alpha=0.3, label='Underfitting / Good Gen')
plt.plot(eval_steps, gap, 'k-', linewidth=2)
plt.title(' Generalization Gap ', fontsize=14)
plt.xlabel('Training Steps')
plt.ylabel('Val Loss - Train Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# --- GRAPH 4: LEARNING VELOCITY (New!) ---
if len(t_loss) > 1:
    velocity = np.diff(smooth(t_loss))
    plt.figure(figsize=(10, 6))
    plt.plot(eval_steps[1:], velocity, 'purple', linewidth=2)
    plt.axhline(0, color='black', linestyle='--')
    plt.title('4. Learning Velocity (Speed of Improvement)', fontsize=14)
    plt.xlabel('Training Steps')
    plt.ylabel('Loss Change per Interval')
    plt.grid(True, alpha=0.3)
    plt.show()

# --- GRAPH 5: LEARNING RATE ---
if len(lrs) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(total_steps, lrs, 'b-', linewidth=2)
    plt.title(' Learning Rate Schedule', fontsize=14)
    plt.xlabel('Steps')
    plt.ylabel('Learning Rate')
    plt.grid(True, alpha=0.3)
    plt.show()

# --- GRAPH 6: GRADIENT NORM (Optional) ---
if len(grads) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(grads, 'gray', alpha=0.4, label='Raw')
    plt.plot(smooth(grads, 0.95), 'k-', linewidth=2, label='Smoothed')
    plt.title(' Gradient Norm (Stability Check)', fontsize=14)
    plt.xlabel('Steps')
    plt.ylabel('Norm Magnitude')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()